# G1 Academy Bonus - Task 1: Using sdk_wrapper.G1 (say / set_headlight)

## Introduction
This is the only notebook in this bonus series that imports directly from `sdk_wrapper.py`. Every later task notebook rebuilds its own slice of the wrapper from native Unitree SDK / DDS calls, the same way `sdk_wrapper.G1` itself was built. Here you instead treat `G1` as a finished component and learn its *usage* surface: construct one `G1` instance per kernel, then call `g1.say(...)` and `g1.set_headlight(...)`.

Why start here? Seeing the finished, polished API first gives a concrete target to rebuild towards in Tasks 2-13. By the time you reach the final task you will have reconstructed enough of `G1` from scratch to know exactly what these two calls do underneath.

## Task 1 - Construct the wrapper once per kernel
`G1.__init__` already calls the module-level `_ensure_factory` guard for you, so you do not need your own `ChannelFactoryInitialize` guard when working through `sdk_wrapper`. It still only tolerates one `(domain_id, iface)` pair per process: construct exactly one `G1` per kernel, and restart the kernel before pointing it at a different interface or domain.

In [ ]:
from sdk_wrapper import G1

# One G1 instance owns the DDS ChannelFactory, every subscriber, and every SDK client for this
# kernel. Re-running this cell is fine (it reuses the module-level ChannelFactoryInitialize
# guard); constructing a *second* G1 with a different iface/domain_id in the same kernel raises.
g1 = G1(iface="eth0", domain_id=0)

## Task 2 - `g1.say(text, language="EN", volume=100)`
`say` synthesizes `text` with Piper, converts the resulting WAV to mono/16-bit/16kHz PCM, and streams it through `AudioClient.PlayStream`. `language` accepts the short codes documented in `sdk_wrapper._PIPER_VOICES` (`en`, `de`, `fr`, `es`, `ar`, plus a few long-form aliases). It raises `ValueError` for an unsupported language and `FileNotFoundError` if the matching Piper voice model is not installed.

In [ ]:
# code = g1.say("G1 academy wrapper ready.", language="EN", volume=80)
# print("say() return code:", code)

## Task 3 - `g1.set_headlight(color="green", intensity=100, duration_s=3)`
`color` accepts a name (`"green"`), a `#RRGGBB` hex string, or an `"R,G,B"` string; `intensity` (0-100) scales brightness before the color is sent. If `duration_s > 0`, `G1` starts a background thread that keeps refreshing `AudioClient.LedControl` with that color every 0.2s until the duration elapses; a new call to `set_headlight` cancels and joins any thread still running from a previous call before starting its own, so repeated calls are last-call-wins. `duration_s <= 0` sends a single `LedControl` call and returns immediately (no thread).

In [ ]:
# code = g1.set_headlight(color="cyan", intensity=60, duration_s=4)
# print("set_headlight() return code:", code)

## Where to go from here
`g1` exposes far more than these two calls: `get_state`/`get_lowstate`/`get_odom`/`get_battery`/`get_slam_info`/`get_service`, `damp_mode`/`prepare_mode`/`walk_mode`/`run_mode`, `toggle_service`/`toggle_gait`, `release_arms`/`engage_arms`, `loco_move`/`loco_stop`, `move_ll_joint`, `open_dex3_hand`/`close_dex3_hand`, `start_mapping`/`stop_mapping`/`relocate`/`navigate`, `clap`/`face_wave`/`shake_hand`, and `ik_move_ee`/`extend_arm`. Tasks 2-13 rebuild each of these pieces natively, one lesson at a time, until you could have written `sdk_wrapper.py` yourself.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.